# SSQ Quant Research System

双色球统计研究、随机基线、频率策略与可选 LSTM 实验。

本 Notebook 不承诺预测中奖号码。所有模型都必须与同成本随机基线比较，并使用严格的时间顺序。

In [ ]:
# Kaggle Notebook 设置
RUN_LSTM = False  # 确认数据格式和基线后再改为 True
WINDOW = 30
SEED = 42
DATA_DIR = '/kaggle/input'
REPO_URL = 'https://github.com/wangmou23/ssq-quant-research-system.git'

In [ ]:
# 获取项目源码；Kaggle 也可以通过 Add data 上传项目压缩包
import os, sys, subprocess, pathlib
repo_dir = pathlib.Path('/kaggle/working/ssq-quant-research-system')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_dir)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))
print(repo_dir)

In [ ]:
# 环境检查
import platform, numpy as np
print('Python:', platform.python_version())
try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
    print('GPU:', tf.config.list_physical_devices('GPU'))
except ImportError:
    print('TensorFlow 未安装；基础回测仍可运行，LSTM 将跳过。')

In [ ]:
# 运行项目自测
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(repo_dir / 'tests')], check=True)

In [ ]:
# 无需外部数据的合成 smoke test
from datetime import date, timedelta
from ssq_research.core import Draw, RandomStrategy, FrequencyStrategy, run_backtest, summarize
start = date(2020, 1, 1)
draws = [Draw(str(2020001+i), (start + timedelta(days=i)).isoformat(), tuple(range(1+i%20, 7+i%20)), 1+i%16) for i in range(80)]
smoke_rows = run_backtest(draws, (RandomStrategy(), FrequencyStrategy()), window=WINDOW, seed=SEED)
summarize(smoke_rows)

In [ ]:
# 在 Kaggle 的 /kaggle/input 中寻找符合格式的 CSV
from pathlib import Path
csv_files = list(Path(DATA_DIR).rglob('*.csv'))
print('发现 CSV 数量:', len(csv_files))
for p in csv_files[:20]:
    print(p)

In [ ]:
# 设置真实数据路径。CSV 必须包含：issue, draw_date, red_1..red_6, blue
DATA_CSV = None  # 例如 '/kaggle/input/your-dataset/ssq.csv'
if DATA_CSV:
    from ssq_research.core import load_csv
    real_draws = load_csv(Path(DATA_CSV))
    print('期数:', len(real_draws), '首期:', real_draws[0].issue, '末期:', real_draws[-1].issue)
else:
    print('尚未设置 DATA_CSV；请先上传数据集并填写路径。')

In [ ]:
# 真实数据基线回测
if DATA_CSV:
    baseline_rows = run_backtest(real_draws, (RandomStrategy(), FrequencyStrategy()), window=WINDOW, seed=SEED)
    baseline_summary = summarize(baseline_rows)
    baseline_summary
else:
    print('跳过：没有 DATA_CSV。')

In [ ]:
# 可选 LSTM：只在确认数据和 GPU 环境后运行
if RUN_LSTM and DATA_CSV:
    from ssq_research.lstm import LstmConfig, walk_forward_lstm
    config = LstmConfig(window=10, hidden_units=32, layers=2, dropout=0.1, epochs=20, batch_size=32)
    lstm_rows = walk_forward_lstm(real_draws, config, min_history=max(WINDOW, config.window + 1), seed=SEED)
    print('LSTM 评估期数:', len(lstm_rows))
    lstm_rows[:3]
elif RUN_LSTM:
    print('RUN_LSTM=True 但 DATA_CSV 未设置，跳过。')
else:
    print('RUN_LSTM=False；当前只运行基线。')

## 解释结果时必须遵守

- 不能把单次回测最高命中当作预测优势。
- 必须同时报告 random、frequency 和 LSTM。
- 必须保留数据来源、时间范围、窗口、随机种子和训练配置。
- LSTM 结果只能说明该配置在历史样本上的模拟表现，不能证明未来中奖概率提高。